# Scale test: control-plane runs over the copied `nwm30_medium_range`

How big a run can a single run job do? Runs one evaluation of `public.nwm30_medium_range`
against `public.usgs_streamflow` (the data `01_copy_configuration_to_public` copied) over
growing windows of forecasts:

| step | forecasts issued | forecast rows (about) |
| --- | --- | --- |
| `day` | 2024-01-01 | 8M |
| `week` | 2024-01-01 .. 01-07 | 55M |
| `month` | January 2024 | 246M |

Each step is its own saved evaluation (`nwm30_mrf_vs_usgs_<step>`) in `PROJECT`, so
their runs and results stay apart. Everything goes through the control plane API as
you. A run's job reads its inputs from the warehouse itself (PyIceberg plans the files,
DuckDB reads them), in shards: ranges of gauges sized to the job's memory
(`JOB_MEMORY_LIMIT`, 4Gi by default), each with its linked forecasts. A run killed for
memory fails with "its pod was killed for running out of memory".

Section 6 shows how many files each shard read. If `file_reads` is near `files`,
each shard's gauges sit in their own files (the table is sorted by series). If it is
near `files × shards`, every shard read every file, and a sorted rewrite of the table
would help.

Self-contained: the specs are built below, nothing is read from files. Works on KinD
too (with `public` holding the copy).

While a run is going, its pod's memory (read-only):

```
kubectl top pod -n teehr-hub -l app=teehr-job
```

In [10]:
import os
import time
from datetime import datetime

import pandas as pd
import requests

## 1. Settings

In [11]:
PROJECT = "scale_tests"  # created below if missing (you own it)
STEPS = ["day", "week", "month"]  # in order; the loop stops at the first failure
POLL_SECONDS = 30

# The control plane API, from inside the cluster (KinD and use1 alike). From outside:
# https://control-plane.teehr.rtiamanzi.org (use1), https://control-plane.teehr.local.app.garden (KinD).
CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed
# The API docs in a browser: a run can be started there when Jupyter's token can't.
DOCS = os.environ.get("CONTROL_PLANE_DOCS", "https://control-plane.teehr.rtiamanzi.org/docs")

In [12]:
def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, ok=(200, 201, 202, 204), **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code not in ok:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:2000]}")
    return resp

## 2. The specs

One evaluation, three windows. Forecasts are filtered on `reference_time` (their
partitions, so the scan skips the rest); observations on `value_time`, from the window's
start to 16 days past its end (the longest lead). Results per gauge and per 1-day
lead-time bin; no pairs written.

In [13]:
WINDOWS = {  # step: (first reference_time, end (exclusive))
    "day": ("2024-01-01", "2024-01-02"),
    "week": ("2024-01-01", "2024-01-08"),
    "month": ("2024-01-01", "2024-02-01"),
}
MAX_LEAD = pd.Timedelta("16D")


def spec(step):
    start, end = (pd.Timestamp(t, tz="UTC") for t in WINDOWS[step])
    at = lambda t: f"'{t:%Y-%m-%dT%H:%M:%SZ}'"  # noqa: E731
    return {
        "version": 1,
        "name": f"nwm30_mrf_vs_usgs_{step}",
        "inputs": {
            "primary": {
                "dataset": "public.usgs_streamflow",
                "filter": f"value_time >= {at(start)} AND value_time < {at(end + MAX_LEAD)}",
            },
            "secondary": [{
                "dataset": "public.nwm30_medium_range",
                "filter": f"reference_time >= {at(start)} AND reference_time < {at(end)}",
            }],
        },
        "pair": {"relation": "public.usgs_nwm30"},
        "fields": [{"name": "lead_time_bin", "fn": "lead_time_bins", "width": "P1D"}],
        "aggregate": [{
            "grouping_sets": [["primary_series_key"], ["lead_time_bin"]],
            "metrics": ["count", "kling_gupta_efficiency", "nash_sutcliffe_efficiency",
                        "root_mean_square_error", "relative_bias", "pearson_correlation"],
        }],
        "output": {"unit": "m3/s"},
    }


SPECS = {step: spec(step) for step in STEPS}
SPECS[STEPS[0]]

{'version': 1,
 'name': 'nwm30_mrf_vs_usgs_day',
 'inputs': {'primary': {'dataset': 'public.usgs_streamflow',
   'filter': "value_time >= '2024-01-01T00:00:00Z' AND value_time < '2024-01-18T00:00:00Z'"},
  'secondary': [{'dataset': 'public.nwm30_medium_range',
    'filter': "reference_time >= '2024-01-01T00:00:00Z' AND reference_time < '2024-01-02T00:00:00Z'"}]},
 'pair': {'relation': 'public.usgs_nwm30'},
 'fields': [{'name': 'lead_time_bin', 'fn': 'lead_time_bins', 'width': 'P1D'}],
 'aggregate': [{'grouping_sets': [['primary_series_key'], ['lead_time_bin']],
   'metrics': ['count',
    'kling_gupta_efficiency',
    'nash_sutcliffe_efficiency',
    'root_mean_square_error',
    'relative_bias',
    'pearson_correlation']}],
 'output': {'unit': 'm3/s'}}

## 3. The project

Created if missing, then provisioned until ready.

In [14]:
if call("GET", f"/projects/{PROJECT}", ok=(200, 404)).status_code == 404:
    call("POST", "/projects", json={
        "name": PROJECT, "display_name": "Scale tests",
        "description": "Control-plane run scale tests over the copied legacy data",
    })
    print(f"created {PROJECT}")
for _ in range(20):
    project = call("GET", f"/projects/{PROJECT}").json()
    if project["status"] == "ready":
        break
    call("POST", f"/projects/{PROJECT}/provision", ok=(200, 202, 409))
    time.sleep(3)
print(f"{PROJECT}: {project['status']}; your role: {project.get('my_role')}")

scale_tests: ready; your role: admin


## 4. Plan (a dry run: saves nothing)

Checks every spec against the datasets as they are: names, roles, units, the crosswalk,
fields and metrics. Fix anything here before running.

In [15]:
for step, s in SPECS.items():
    resp = call("POST", f"/projects/{PROJECT}/runs/plan", ok=(200, 422), json=s)
    body = resp.json()
    if resp.status_code == 422:
        print(f"{step}: does not plan: {body.get('errors') or body}")
        continue
    inputs = ", ".join(f"{i['dataset']} ({i['unit']})" for i in body["inputs"])
    print(f"{step}: plans; unit {body['unit']}; inputs: {inputs}")

day: plans; unit m3/s; inputs: public.usgs_streamflow (m3/s), public.nwm30_medium_range (m3/s)
week: plans; unit m3/s; inputs: public.usgs_streamflow (m3/s), public.nwm30_medium_range (m3/s)
month: plans; unit m3/s; inputs: public.usgs_streamflow (m3/s), public.nwm30_medium_range (m3/s)


## 5. Run

Each step in turn: save its evaluation, start a run, wait for it. Stops at the first
failure (a step bigger than one that failed would fail too). A step whose latest
version already succeeded is skipped, so the cell can be run again after a failure.

In [20]:
def seconds(a, b):
    return (datetime.fromisoformat(b) - datetime.fromisoformat(a)).total_seconds() if a and b else None


def latest_run(slug):
    runs = call("GET", f"/projects/{PROJECT}/evaluations/{slug}/runs").json()
    return runs[0] if runs else None


def run_step(step):
    slug = SPECS[step]["name"]
    saved = call("PUT", f"/projects/{PROJECT}/evaluations/{slug}", json=SPECS[step]).json()
    last = latest_run(slug)
    if last and last["status"] == "succeeded" and last["version"] == saved["version"]:
        print(f"{step}: version {saved['version']} already ran")
        return last
    if last and last["status"] in ("queued", "running"):
        run = last
        print(f"{step}: waiting for run {run['id']}")
    else:
        resp = call("POST", f"/projects/{PROJECT}/evaluations/{slug}/runs", ok=(202, 502))
        if resp.status_code == 502:
            print(f"{step}: {resp.json()['detail']}\n"
                  f"  Your Keycloak browser session has likely ended (Jupyter keeps going on an\n"
                  f"  offline token, which a job can't act as). Start the run from the API docs\n"
                  f"  ({DOCS}: sign in, then POST /v1/projects/{PROJECT}/evaluations/{slug}/runs)\n"
                  f"  and run this cell again: it waits for that run.")
            return None
        run = resp.json()
        print(f"{step}: started run {run['id']} (version {saved['version']})")
    began = time.monotonic()
    while run["status"] in ("queued", "running"):
        time.sleep(POLL_SECONDS)
        run = call("GET", f"/projects/{PROJECT}/runs/{run['id']}").json()
        print(f"  {time.monotonic() - began:6.0f}s {run['status']}", flush=True)
    return run


summary = []
for step in STEPS:
    run = run_step(step)
    if run is None:
        break
    result = run.get("result") or {}
    summary.append({
        "step": step, "status": run["status"],
        "queued_s": seconds(run["created_at"], run["started_at"]),
        "ran_s": seconds(run["started_at"], run["finished_at"]),
        "mode": result.get("mode"), "shards": result.get("shards"),
        "reads": result.get("reads"), "results_dataset": run["results_dataset"],
        "error": run["error"],
    })
    print(f"{step}: {run['status']}" + (f": {run['error']}" if run["error"] else ""))
    print(f"  result: {result}")
    if run["status"] != "succeeded":
        break
pd.DataFrame(summary).drop(columns="reads", errors="ignore")

day: version 1 already ran
day: succeeded
  result: {'dataset': 'results_nwm30_mrf_vs_usgs_day_v1', 'rows': 6654, 'snapshot_id': '1819567130781077969', 'state': {'dataset': 'state_nwm30_mrf_vs_usgs_day_v1', 'rows': 6654, 'snapshot_id': '7304770064913099176', 'bucket': 'month'}, 'inputs': {'public.usgs_streamflow': 5438162374574478200, 'public.usgs_streamflow_series': 516669140977086668, 'public.nwm30_medium_range': 8671330660241011713, 'public.nwm30_medium_range_series': 2270510137369578091, 'public.usgs_nwm30': 7199196646887357891}, 'shards': 144, 'reads': {'public.usgs_nwm30': {'reads': 1, 'files': 1, 'file_reads': 1, 'bytes_read': 66194}, 'public.usgs_streamflow': {'reads': 144, 'files': 23, 'file_reads': 166, 'bytes_read': 56118348}, 'public.usgs_streamflow_series': {'reads': 144, 'files': 1, 'file_reads': 144, 'bytes_read': 8690256}, 'public.nwm30_medium_range': {'reads': 144, 'files': 22, 'file_reads': 735, 'bytes_read': 10885155110}, 'public.nwm30_medium_range_series': {'reads':

,step,status,queued_s,ran_s,mode,shards,results_dataset,error
0,day,succeeded,3.823193,193.020891,full,144,results_nwm30_mrf_vs_usgs_day_v1,None
1,week,succeeded,4.177960,266.821786,full,144,results_nwm30_mrf_vs_usgs_week_v1,None
2,month,succeeded,4.155208,572.751915,full,144,results_nwm30_mrf_vs_usgs_month_v1,None


## 6. Files read per table

Per step and table: the reads (one per shard, plus whole-table reads), the distinct files
they planned, the file reads that took, and the overlap (`file_reads / files`): 1 means
each file was read once, the shard count means every shard read every file.

In [21]:
rows = []
for row in summary:
    for table, r in (row.get("reads") or {}).items():
        rows.append({"step": row["step"], "shards": row["shards"], "table": table, **r,
                     "overlap": round(r["file_reads"] / r["files"], 2) if r["files"] else None,
                     "gb_read": round(r["bytes_read"] / 1e9, 2)})
pd.DataFrame(rows)

,step,shards,table,reads,files,file_reads,bytes_read,overlap,gb_read
0,day,144,public.usgs_nwm30,1,1,1,66194,1.00,0.00
1,day,144,public.usgs_streamflow,144,23,166,56118348,7.22,0.06
2,day,144,public.usgs_streamflow_series,144,1,144,8690256,144.00,0.01
3,day,144,public.nwm30_medium_range,144,22,735,10885155110,33.41,10.89
4,day,144,public.nwm30_medium_range_series,144,1,144,5050368,144.00,0.01
5,week,144,public.usgs_nwm30,1,1,1,66194,1.00,0.00
6,week,144,public.usgs_streamflow,144,23,166,56118348,7.22,0.06
7,week,144,public.usgs_streamflow_series,144,1,144,8690256,144.00,0.01
8,week,144,public.nwm30_medium_range,144,22,735,10885155110,33.41,10.89
9,week,144,public.nwm30_medium_range_series,144,1,144,5050368,144.00,0.01


## 7. A look at the results

In [19]:
for row in summary:
    if row["status"] != "succeeded":
        continue
    data = call("GET", f"/projects/{PROJECT}/datasets/{row['results_dataset']}/data",
                params={"limit": 10}).json()
    print(f"{row['step']}: {row['results_dataset']}")
    display(pd.DataFrame(data["rows"]))

day: results_nwm30_mrf_vs_usgs_day_v1


,primary_series_key,lead_time_bin,grouping,count,kling_gupta_efficiency,nash_sutcliffe_efficiency,root_mean_square_error,relative_bias,pearson_correlation,run_id,spec_version,engine_version
0,(all),"[0h, 24h)",lead_time_bin,573979,0.836258,0.826566,104.745649,-0.034463,0.910170,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
1,(all),"[24h, 48h)",lead_time_bin,595324,0.808149,0.777941,121.849923,-0.038435,0.882509,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
2,(all),"[48h, 72h)",lead_time_bin,595616,0.808364,0.759806,128.216254,-0.033531,0.871738,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
3,(all),"[72h, 96h)",lead_time_bin,596782,0.814724,0.760311,127.458288,-0.018380,0.871973,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
4,(all),"[96h, 120h)",lead_time_bin,594614,0.815714,0.757922,127.448455,-0.004994,0.870589,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
5,(all),"[120h, 144h)",lead_time_bin,591286,0.806512,0.746272,131.336559,-0.001281,0.863872,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
6,(all),"[144h, 168h)",lead_time_bin,585052,0.800554,0.742197,133.557824,-0.017814,0.861520,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
7,(all),"[168h, 192h)",lead_time_bin,578670,0.772703,0.745574,135.075493,-0.048885,0.864423,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
8,(all),"[192h, 216h)",lead_time_bin,573502,0.610852,0.730231,154.174928,-0.271650,0.862231,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1
9,(all),"[216h, 240h)",lead_time_bin,573618,0.468652,0.677335,205.179714,-0.376447,0.848736,98d2b8d9-c316-49b3-ac51-f2e8bdbc26fc,1,teehr-core 0.0.1


week: results_nwm30_mrf_vs_usgs_week_v1


,primary_series_key,lead_time_bin,grouping,count,kling_gupta_efficiency,nash_sutcliffe_efficiency,root_mean_square_error,relative_bias,pearson_correlation,run_id,spec_version,engine_version
0,(all),"[0h, 24h)",lead_time_bin,3983756,0.861461,0.828797,107.328113,-0.031123,0.910464,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
1,(all),"[24h, 48h)",lead_time_bin,4137344,0.819994,0.764513,126.937627,-0.049646,0.874411,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
2,(all),"[48h, 72h)",lead_time_bin,4115522,0.799767,0.753454,132.717062,-0.094812,0.868229,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
3,(all),"[72h, 96h)",lead_time_bin,4093524,0.722094,0.739649,144.629788,-0.180217,0.860653,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
4,(all),"[96h, 120h)",lead_time_bin,4065918,0.660876,0.726085,160.610887,-0.228017,0.854908,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
5,(all),"[120h, 144h)",lead_time_bin,4029397,0.619520,0.720497,175.213114,-0.250928,0.855964,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
6,(all),"[144h, 168h)",lead_time_bin,3982158,0.575577,0.711987,193.603551,-0.276139,0.857608,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
7,(all),"[168h, 192h)",lead_time_bin,3917843,0.529857,0.697630,217.172522,-0.298818,0.858581,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
8,(all),"[192h, 216h)",lead_time_bin,3834124,0.506760,0.694287,235.101074,-0.307987,0.864592,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1
9,(all),"[216h, 240h)",lead_time_bin,3741558,0.500550,0.690924,253.479638,-0.297289,0.868585,d1d6a79b-4ddd-446f-98fc-534b4d206e1c,1,teehr-core 0.0.1


## 8. Clean up (optional)

Deletes the step evaluations with their runs and results. The project stays.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for step in STEPS:
        slug = SPECS[step]["name"]
        call("DELETE", f"/projects/{PROJECT}/evaluations/{slug}", ok=(200, 204, 404))
        print(f"deleted {slug}")